# Portfolio Demo Report Viewer

This notebook loads a portfolio demo JSON report and displays:
- Run summary
- Portfolio metrics
- Top alpha scores
- Target weights
- Rebalance order sample

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Update this path if you generated a different report filename
REPORT_PATH = Path('../eval_results/portfolio/demo_run_2026-02-28.json')
assert REPORT_PATH.exists(), f'Report not found: {REPORT_PATH.resolve()}'

report = json.loads(REPORT_PATH.read_text())
report.keys()

In [ ]:
summary = {
    'run_at_utc': report.get('run_at_utc'),
    'trade_date': report.get('trade_date'),
    'symbol_count': report.get('symbol_count'),
    'benchmark_symbol': report.get('benchmark_symbol'),
    'rebalance_orders_count': report.get('rebalance_orders_count'),
}
pd.DataFrame([summary])

In [ ]:
metrics_df = pd.DataFrame([report.get('portfolio_metrics', {})]).T
metrics_df.columns = ['value']
metrics_df

In [ ]:
alpha_df = (
    pd.DataFrame(list(report.get('top_alpha_scores', {}).items()), columns=['symbol', 'alpha'])
    .sort_values('alpha', ascending=False)
    .reset_index(drop=True)
)
alpha_df

In [ ]:
if not alpha_df.empty:
    ax = alpha_df.plot(kind='bar', x='symbol', y='alpha', figsize=(10, 4), legend=False, title='Top Alpha Scores')
    ax.set_ylabel('alpha score')
    plt.tight_layout()
    plt.show()

In [ ]:
weights_df = (
    pd.DataFrame(list(report.get('target_weights', {}).items()), columns=['symbol', 'weight'])
    .sort_values('weight', ascending=False)
    .reset_index(drop=True)
)
weights_df.head(20)

In [ ]:
top_w = weights_df.head(20).iloc[::-1]  # plot largest 20
if not top_w.empty:
    ax = top_w.plot(kind='barh', x='symbol', y='weight', figsize=(8, 6), legend=False, title='Top 20 Target Weights')
    ax.set_xlabel('weight')
    plt.tight_layout()
    plt.show()

In [ ]:
orders_df = pd.DataFrame(report.get('rebalance_orders_sample', []))
orders_df